# Minería de Datos — Evaluación 1
## Análisis Exploratorio: Steam Games Dataset

| | |
|---|---|
| **Alumno** | Amaro Araneda |
| **Alumno** | Josseph Valverde |
| **Asignatura** | Minería de Datos |
| **Sección** | BIY7121 |
| **Fecha** | Abril 2026 |
| **Dataset** | Steam Games Dataset |
| **Fuente** | Kaggle — https://www.kaggle.com/datasets/nikdavis/steam-store-games |

---

## Justificación del Dataset

El dataset seleccionado contiene información de más de **27.000 videojuegos** publicados en la plataforma **Steam** (la tienda digital de videojuegos para PC más grande del mundo), recopilada a través de la API oficial de Steam y SteamSpy.

Se escogió este dataset porque:
- Tiene más de 27.000 filas, lo que lo hace amplio para minería de datos.
- Contiene variables **numéricas** (precio, ratings, tiempo de juego, logros) y **categóricas** (géneros, plataformas, desarrollador), cumpliendo el requisito de diversidad.
- Permite analizar un fenómeno económico y social relevante: el mercado de videojuegos digitales.
- Es adecuado para futuras aplicaciones de modelos de **clasificación** (predecir si un juego tendrá buena recepción) o **regresión** (predecir el precio o cantidad de propietarios).

---
## 1. Importación de Librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

**Justificación:** Se importan las tres librerías fundamentales del análisis de datos en Python: `pandas` para manipular los datos en forma de tabla (DataFrame), `numpy` para operaciones matemáticas y `matplotlib.pyplot` para generar gráficos.

---
## 2. Carga de Datos

In [ ]:
# Carga de datos
# Subir el archivo steam.csv desde el panel de archivos de Colab antes de ejecutar
data_frame = pd.read_csv('steam.csv')

# Separar la columna 'platforms' en tres columnas booleanas independientes
data_frame['windows'] = data_frame['platforms'].str.contains('windows', case=False, na=False)
data_frame['mac']     = data_frame['platforms'].str.contains('mac',     case=False, na=False)
data_frame['linux']   = data_frame['platforms'].str.contains('linux',   case=False, na=False)

# Eliminar la columna original 'platforms' (ya está representada por las tres columnas booleanas)
data_frame = data_frame.drop(columns=['platforms'])

data_frame.head()

**Justificación:** Se utiliza `pd.read_csv()` para cargar el archivo CSV en un DataFrame. Inmediatamente después, la columna `platforms` (que contenía valores como `'windows;mac;linux'`) se descompone en tres columnas booleanas separadas: `windows`, `mac` y `linux`. Cada una indica con `True`/`False` si el juego es compatible con esa plataforma. La columna original `platforms` se elimina para evitar redundancia. La función `.head()` muestra las primeras 5 filas del dataset.

**Interpretación:** Cada fila representa un videojuego publicado en Steam. Las nuevas columnas `windows`, `mac` y `linux` permiten filtrar y analizar compatibilidad de plataformas de forma directa y eficiente, sin necesidad de parsear texto.

---
## 3. Tamaño del Dataset

In [ ]:
# Conocer la cantidad de observaciones y la cantidad de características
data_frame.shape

**Justificación:** La propiedad `.shape` retorna una tupla con el número de filas y columnas del DataFrame.

**Interpretación:** El dataset original cuenta con **27.075 observaciones (juegos)**. Tras separar la columna `platforms` en tres columnas booleanas (`windows`, `mac`, `linux`) y eliminar la columna original, el dataset ahora tiene **20 columnas** (18 originales − 1 `platforms` + 3 nuevas). Este cambio mejora la usabilidad de las columnas de plataforma para análisis y modelos.

---
## 4. Mapeo de Datos

El mapeo de datos consiste en documentar el significado, tipo y descripción de cada columna del dataset. Es una buena práctica antes de comenzar cualquier análisis.

In [ ]:
# Mapeo de datos: diccionario de columnas del dataset
mapeo = pd.DataFrame({
    'Columna': [
        'appid', 'name', 'release_date', 'english', 'developer',
        'publisher', 'required_age', 'categories',
        'genres', 'steamspy_tags', 'achievements',
        'positive_ratings', 'negative_ratings',
        'average_playtime', 'median_playtime',
        'owners', 'price',
        'windows', 'mac', 'linux'
    ],
    'Tipo de Variable': [
        'Numérica', 'Categórica', 'Categórica', 'Numérica',
        'Categórica', 'Categórica', 'Numérica',
        'Categórica', 'Categórica', 'Categórica',
        'Numérica', 'Numérica', 'Numérica',
        'Numérica', 'Numérica',
        'Categórica', 'Numérica',
        'Numérica', 'Numérica', 'Numérica'
    ],
    'Tipo de Dato': [
        'int64', 'object', 'object', 'int64',
        'object', 'object', 'int64',
        'object', 'object', 'object',
        'int64', 'int64', 'int64',
        'int64', 'int64',
        'object', 'float64',
        'bool', 'bool', 'bool'
    ],
    'Descripción': [
        'Identificador único del juego en Steam',
        'Nombre del videojuego',
        'Fecha de lanzamiento',
        'Disponible en inglés (1=Sí, 0=No)',
        'Empresa o persona que desarrolló el juego',
        'Empresa que publicó el juego',
        'Edad mínima requerida (0 = sin restricción)',
        'Categorías de funcionalidad del juego',
        'Géneros del juego (Acción, RPG, etc.)',
        'Etiquetas asignadas por la comunidad',
        'Cantidad de logros disponibles',
        'Total de reseñas positivas en Steam',
        'Total de reseñas negativas en Steam',
        'Tiempo promedio de juego (en minutos)',
        'Tiempo mediano de juego (en minutos)',
        'Rango estimado de propietarios del juego',
        'Precio del juego en dólares USD',
        'Compatible con Windows (True/False)',
        'Compatible con macOS (True/False)',
        'Compatible con Linux (True/False)'
    ]
})

mapeo

**Justificación:** Se construye manualmente un DataFrame con la descripción de cada columna. Esta práctica es fundamental para entender el contexto del negocio antes de analizar los datos.

**Interpretación:** El dataset tiene **11 variables numéricas** y **9 variables categóricas** (total 20 columnas). Las columnas `windows`, `mac` y `linux` son de tipo `bool` y reemplazan la columna textual `platforms`. Las variables numéricas más importantes para el análisis son `positive_ratings`, `negative_ratings`, `price` y `average_playtime`. La columna `owners` sigue siendo categórica porque contiene rangos de texto (ej.: '10000000-20000000').

---
## 5. Tipos de Datos de las Columnas

In [ ]:
# Revisión de los tipos de datos de cada columna
data_frame.dtypes

**Justificación:** La propiedad `.dtypes` muestra el tipo de dato que Python asignó automáticamente a cada columna al cargar el archivo CSV (y tras la transformación de plataformas).

**Interpretación:** Las columnas numéricas tienen tipo `int64` o `float64`, mientras que las columnas de texto aparecen como `object`. Las columnas `windows`, `mac` y `linux` aparecen como `bool`, ya que fueron generadas con `.str.contains()`. La columna `owners` sigue siendo `object` porque contiene rangos en texto.

In [ ]:
# Agrupación de las columnas según su tipo de dato
tipos = data_frame.columns.to_series().groupby(data_frame.dtypes).groups

# Columnas categóricas (tipo object)
columnas_categoricas = list(tipos.get(np.dtype('object'), []))
print('Columnas categóricas:', len(columnas_categoricas))
print(columnas_categoricas)

print()

# Columnas booleanas (tipo bool) — plataformas
columnas_booleanas = list(tipos.get(np.dtype('bool'), []))
print('Columnas booleanas (plataformas):', len(columnas_booleanas))
print(columnas_booleanas)

print()

# Columnas numéricas (el resto: int64, float64)
columnas_numericas = list(set(data_frame.columns) - set(columnas_categoricas) - set(columnas_booleanas))
print('Columnas numéricas:', len(columnas_numericas))
print(columnas_numericas)

**Justificación:** Se agrupa las columnas según su tipo de dato usando `.groupby()` sobre los dtypes. Se identifican tres grupos: categóricas (`object`), booleanas (`bool`) y numéricas (`int64`/`float64`).

**Interpretación:** Hay **9 columnas categóricas** (tipo `object`), **3 columnas booleanas** (`windows`, `mac`, `linux`) y **8 columnas numéricas**. Las columnas booleanas se tratarán de forma independiente en la imputación (rellenando con `False` si hubiese nulos), mientras que las numéricas se imputan con la media y las categóricas con la moda.

---
## 6. Tratamiento de Valores Nulos

In [ ]:
# Verificación de si existe algún valor nulo en el dataset
data_frame.isnull().any().any()

**Justificación:** `.isnull().any().any()` retorna `True` si existe al menos un valor nulo en todo el DataFrame, o `False` si el dataset está completo.

**Interpretación:** Si el resultado es `False`, el dataset no tiene valores nulos y no es necesario aplicar técnicas de imputación. Si es `True`, se deben tratar los nulos antes de continuar.

In [ ]:
# Detalle de valores nulos por columna
for feature in data_frame.columns:
    print('Total de valores nulos de', feature, '=', data_frame[feature].isna().sum())

**Justificación:** Se recorre cada columna con un loop `for` y se cuenta la cantidad de valores nulos usando `.isna().sum()`. Esto permite ver de forma precisa en qué columnas hay datos faltantes y cuántos.

**Interpretación:** Si todas las columnas muestran 0 valores nulos, el dataset está completo y no requiere limpieza. De lo contrario, las columnas con nulos deben ser tratadas mediante imputación (rellenar con la media o la moda) o eliminación de registros.

In [ ]:
# Técnica 1: Imputación de valores nulos en columnas numéricas (con la media)
for columna in columnas_numericas:
    media = data_frame[columna].mean()
    data_frame[columna] = data_frame[columna].fillna(media)

# Técnica 2: Imputación de valores nulos en columnas categóricas (con la moda)
for columna in columnas_categoricas:
    moda = data_frame[columna].mode()[0]
    data_frame[columna] = data_frame[columna].fillna(moda)

# Técnica 3: Imputación de valores nulos en columnas booleanas (con False)
# Si un juego no tiene plataforma registrada, se asume incompatibilidad
for columna in columnas_booleanas:
    data_frame[columna] = data_frame[columna].fillna(False)

# Verificación: comprobar que ya no hay nulos
print('¿Quedan valores nulos?', data_frame.isnull().any().any())

**Justificación:** Se aplican tres técnicas de imputación según el tipo de columna:
- Para columnas **numéricas**: se rellena con la **media**, que es el valor más representativo.
- Para columnas **categóricas**: se rellena con la **moda** (valor más frecuente).
- Para columnas **booleanas** (`windows`, `mac`, `linux`): se rellena con `False`, asumiendo que si no hay información de compatibilidad, el juego no está disponible en esa plataforma.

**Interpretación:** Después de aplicar las tres técnicas de imputación, el resultado `False` confirma que no quedan valores nulos en el dataset. El dataset está listo para el análisis estadístico.

---
## 7. Estadísticas Descriptivas

In [ ]:
# Estadísticas descriptivas de las columnas numéricas
data_frame.describe()

**Justificación:** `.describe()` calcula automáticamente las principales métricas estadísticas para las columnas numéricas: cantidad de valores (`count`), promedio (`mean`), desviación estándar (`std`), mínimo (`min`), cuartiles (25%, 50%, 75%) y máximo (`max`).

**Interpretación:**
- El **precio** (`price`) varía entre 0 (juegos gratuitos) y valores altos. El promedio es bajo, lo que indica que la mayoría de juegos son económicos.
- Los **ratings positivos** tienen una desviación estándar muy alta, lo que indica que la diferencia entre juegos populares y desconocidos es enorme.
- El **tiempo de juego promedio** (`average_playtime`) tiene muchos valores en 0, lo que sugiere que muchos juegos casi no son jugados.
- La **desviación estándar** indica que los datos están muy dispersos en casi todas las columnas numéricas.

In [ ]:
# Estadísticas incluyendo también las columnas categóricas
data_frame.describe(include='all')

**Justificación:** `describe(include='all')` extiende el análisis para incluir también las columnas de tipo `object`, mostrando la cantidad de valores únicos (`unique`), el valor más frecuente (`top`) y su frecuencia (`freq`).

**Interpretación:** Para las columnas categóricas podemos ver, por ejemplo, cuál es el desarrollador más frecuente en el dataset y cuántos juegos tiene publicados. Esto permite identificar los actores más relevantes dentro del mercado de Steam.

In [ ]:
# Estadísticas individuales de la columna precio
print('Mínimo:', data_frame['price'].min())
print('Máximo:', data_frame['price'].max())
print('Promedio:', data_frame['price'].mean())
print('STD:', data_frame['price'].std())
print(data_frame['price'].quantile([.25, .5, .75]))

**Justificación:** Se calculan las métricas estadísticas individualmente para la columna `price` usando los métodos `.min()`, `.max()`, `.mean()`, `.std()` y `.quantile()`.

**Interpretación:** La diferencia entre la mediana (50%) y el máximo del precio es enorme, lo que confirma la existencia de valores atípicos (juegos muy caros). La mayoría de los juegos tienen un precio menor a lo que indica la media, porque unos pocos juegos de precio muy alto elevan ese promedio.

---
## 8. Agrupaciones y Exploración por Categorías

In [ ]:
# Distribución de juegos por plataforma (columnas booleanas)
conteo_plataformas = pd.Series({
    'Windows': data_frame['windows'].sum(),
    'Mac':     data_frame['mac'].sum(),
    'Linux':   data_frame['linux'].sum()
})

print('Juegos compatibles por plataforma:')
print(conteo_plataformas)
print()

# Compatibilidad combinada: juegos disponibles en las tres plataformas
todas = data_frame[data_frame['windows'] & data_frame['mac'] & data_frame['linux']]
solo_windows = data_frame[data_frame['windows'] & ~data_frame['mac'] & ~data_frame['linux']]
print(f'Juegos en las 3 plataformas: {len(todas)}')
print(f'Juegos solo en Windows:       {len(solo_windows)}')

**Justificación:** Dado que `platforms` fue separada en tres columnas booleanas, ahora es posible contar directamente cuántos juegos son compatibles con cada sistema operativo usando `.sum()` (que cuenta los `True`). Además, se pueden combinar las columnas con operadores lógicos (`&`, `~`) para filtrar juegos según compatibilidad múltiple.

**Interpretación:** La gran mayoría de los juegos en Steam son compatibles con **Windows**, lo que confirma que es la plataforma dominante. Un subconjunto significativo también soporta **Mac** y **Linux**, aunque en menor proporción. Los juegos disponibles en las tres plataformas representan el esfuerzo de los desarrolladores por alcanzar una audiencia más amplia.

In [ ]:
# Top 10 géneros más populares
generos = data_frame['genres'].str.split(';').explode()
generos.value_counts().head(10)

**Justificación:** La columna `genres` contiene múltiples géneros separados por punto y coma para cada juego. Se usa `.str.split(';').explode()` para separar cada género en filas individuales y luego `.value_counts()` para contar las ocurrencias.

**Interpretación:** El género **Indie** es el más común en Steam, lo que refleja que la plataforma es un punto de entrada fundamental para desarrolladores independientes. Le siguen **Acción** y **Casual**, géneros con alta demanda entre los jugadores.

In [ ]:
# Juegos con el mayor número de ratings positivos
data_frame[['name', 'positive_ratings', 'negative_ratings', 'price']].sort_values(
    'positive_ratings', ascending=False).head(10)

**Justificación:** Se filtra el DataFrame para mostrar solo las columnas relevantes, y se ordena por `positive_ratings` de mayor a menor con `.sort_values()`.

**Interpretación:** Los juegos más valorados positivamente son títulos muy conocidos como Counter-Strike y DOTA 2. Estos juegos tienen millones de reseñas, lo que los convierte en outliers dentro del dataset. Curiosamente, muchos de los más valorados son gratuitos (precio 0).

---
## 9. Visualizaciones

In [ ]:
# Histograma de la distribución de precios (excluyendo gratuitos para mejor visualización)
precios = data_frame[data_frame['price'] > 0]['price']
precios[precios <= 60].hist(bins=30)
plt.title('Distribución de Precios de Juegos en Steam')
plt.xlabel('Precio (USD)')
plt.ylabel('Cantidad de Juegos')
plt.show()

**Justificación:** Se usa `.hist()` de pandas para generar un histograma. Se excluyen los juegos gratuitos para observar mejor la distribución de los precios de pago. Se filtra hasta $60 para evitar que los outliers compriman el gráfico.

**Interpretación:** La distribución del precio está **fuertemente sesgada hacia la izquierda**: la gran mayoría de los juegos tiene un precio entre $1 y $20. Los juegos más caros son una minoría. Esto indica que el mercado de Steam está dominado por productos de precio accesible.

In [ ]:
# Gráfico de barras: Top 10 géneros más frecuentes
top_generos = generos.value_counts().head(10)
x_values = top_generos.index.tolist()
y_values = top_generos.values.tolist()

plt.bar(x_values, y_values)
plt.title('Top 10 Géneros más Frecuentes en Steam')
plt.xlabel('Género')
plt.ylabel('Cantidad de Juegos')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

**Justificación:** Se usa `plt.bar()` para generar un gráfico de barras con los 10 géneros más frecuentes. `.xticks(rotation=45)` rota las etiquetas del eje X para que se lean correctamente.

**Interpretación:** El género Indie domina ampliamente, seguido de Acción y Casual. Esto refleja la democratización del desarrollo de videojuegos: hoy cualquier persona puede publicar un juego en Steam, lo que ha impulsado la categoría Indie por sobre géneros más tradicionales.

In [ ]:
# Gráfico de torta: distribución de juegos gratuitos vs de pago
tipos_precio = ['Gratuito', 'De Pago']
total_precio = [
    data_frame[data_frame['price'] == 0]['price'].count(),
    data_frame[data_frame['price'] > 0]['price'].count()
]
explode = [0.05, 0]

plt.pie(total_precio, labels=tipos_precio, explode=explode, autopct='%1.1f%%',
        shadow=True, startangle=90)
plt.title('Distribución de Juegos Gratuitos vs De Pago')
plt.show()

**Justificación:** Se usa `plt.pie()` para mostrar la proporción de juegos gratuitos versus de pago. El parámetro `explode` separa ligeramente el segmento de juegos gratuitos para destacarlo visualmente.

**Interpretación:** La mayoría de los juegos en Steam son de pago. Sin embargo, el porcentaje de juegos gratuitos no es despreciable, especialmente considerando que los juegos gratuitos (como DOTA 2 o Counter-Strike) suelen ser los más populares y con más jugadores activos.

In [ ]:
# Gráfico de dispersión: ratings positivos vs negativos
muestra = data_frame[data_frame['positive_ratings'] <= 50000]
x_values = muestra['positive_ratings'].tolist()
y_values = muestra['negative_ratings'].tolist()

plt.scatter(x_values, y_values, marker='o', alpha=0.3, s=5)
plt.title('Relación entre Ratings Positivos y Negativos')
plt.xlabel('Ratings Positivos')
plt.ylabel('Ratings Negativos')
plt.show()

**Justificación:** Se usa `plt.scatter()` para visualizar la relación entre los ratings positivos y negativos. Se filtra a juegos con menos de 50.000 ratings positivos para evitar que los outliers distorsionen el gráfico.

**Interpretación:** Se observa una tendencia positiva: los juegos con más ratings positivos también tienden a tener más ratings negativos. Esto no significa que sean malos juegos, sino que simplemente tienen más jugadores en total, y con más jugadores hay más votos en general.

In [ ]:
# Boxplot del precio (sin outliers extremos) para ver la distribución
precios_bp = data_frame[data_frame['price'] <= 60]['price'].tolist()
plt.boxplot(precios_bp)
plt.title('Distribución del Precio de Juegos en Steam')
plt.ylabel('Precio (USD)')
plt.show()

**Justificación:** El `boxplot` (gráfico de caja y bigote) es útil para visualizar la distribución de una variable numérica e identificar valores atípicos (outliers), que aparecen como puntos fuera de los bigotes del gráfico.

**Interpretación:** El gráfico muestra que el 50% de los juegos tienen un precio entre aproximadamente $0 y $15. Los puntos sobre el bigote superior representan juegos con precios considerablemente más altos que el promedio, que serían valores atípicos a tener en cuenta si se va a entrenar un modelo predictivo.

In [ ]:
# Segmentación por rango de precio usando pd.cut()
data_frame_precios = data_frame[data_frame['price'] > 0].copy()
rango_precios = pd.cut(data_frame_precios['price'], [0, 5, 15, 30, 60, 200])

plot = pd.value_counts(rango_precios).plot(kind='bar',
    title='Cantidad de Juegos por Rango de Precio')
plot.set_ylabel('Cantidad de Juegos')
plot.set_xlabel('Rango de Precio (USD)')
plt.tight_layout()
plt.show()

**Justificación:** `pd.cut()` permite segmentar una variable numérica continua en rangos o categorías. Luego `pd.value_counts().plot(kind='bar')` genera el gráfico de barras directamente desde pandas.

**Interpretación:** El rango de precio entre $5 y $15 concentra la mayor cantidad de juegos de pago en Steam. Esto confirma que los desarrolladores prefieren precios accesibles para maximizar las ventas. Los juegos con precio superior a $60 son una rareza.

In [ ]:
# Histograma de la distribución de logros (achievements) por juego
data_frame[data_frame['achievements'] > 0]['achievements'].hist(bins=40)
plt.title('Distribución de Logros (Achievements) por Juego')
plt.xlabel('Cantidad de Logros')
plt.ylabel('Frecuencia')
plt.show()

**Justificación:** Se filtra para mostrar solo los juegos que tienen al menos un logro, y se usa `.hist()` para ver cómo se distribuye la cantidad de logros por juego.

**Interpretación:** La mayoría de los juegos que implementan logros tienen entre 1 y 100. La distribución está también sesgada hacia la izquierda: son pocos los juegos que tienen cientos o miles de logros disponibles.

---
## 10. Análisis de Correlación

In [ ]:
# Seleccionamos solo las columnas numéricas relevantes para la correlación
columnas_corr = ['required_age', 'achievements', 'positive_ratings',
                 'negative_ratings', 'average_playtime', 'median_playtime', 'price']

# Calculamos la matriz de correlación
correlacion = data_frame[columnas_corr].corr()
correlacion

**Justificación:** El método `.corr()` calcula la **correlación de Pearson** entre todas las columnas numéricas seleccionadas. El resultado es una matriz donde cada celda indica el grado de relación lineal entre dos variables. Los valores van de **-1** (correlación negativa perfecta) a **+1** (correlación positiva perfecta), siendo **0** ausencia de correlación.

**Interpretación:**
- `positive_ratings` y `negative_ratings` tienen una **correlación positiva alta**: los juegos con más reseñas positivas también tienen más negativas, simplemente porque son más populares y tienen más jugadores en total.
- `average_playtime` y `median_playtime` también muestran **correlación positiva**, ya que ambas miden el tiempo de juego.
- `price` no muestra correlación fuerte con los ratings, lo que indica que el precio de un juego no determina si será bien o mal recibido.

In [ ]:
# Visualización de la correlación como mapa de calor
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(10, 7))

# Crear el mapa de calor con matplotlib puro
im = ax.imshow(correlacion.values, cmap='RdYlGn', vmin=-1, vmax=1)

# Etiquetas de ejes
ax.set_xticks(np.arange(len(columnas_corr)))
ax.set_yticks(np.arange(len(columnas_corr)))
ax.set_xticklabels(columnas_corr, rotation=45, ha='right')
ax.set_yticklabels(columnas_corr)

# Anotar cada celda con el valor de correlación
for i in range(len(columnas_corr)):
    for j in range(len(columnas_corr)):
        ax.text(j, i, f'{correlacion.values[i, j]:.2f}',
                ha='center', va='center', fontsize=9, color='black')

plt.colorbar(im, ax=ax)
plt.title('Mapa de Correlación — Variables Numéricas Steam Dataset')
plt.tight_layout()
plt.show()

**Justificación:** Se usa `imshow()` de matplotlib para graficar la matriz de correlación como un mapa de calor. La escala de color `RdYlGn` (rojo-amarillo-verde) facilita la lectura: verde indica correlación positiva alta, rojo indica correlación negativa alta y amarillo indica correlación cercana a 0. Se anotan los valores numéricos en cada celda para mayor precisión.

**Interpretación:** El mapa de calor confirma visualmente lo analizado en la tabla de correlación. La diagonal siempre es 1.0 (una variable siempre se correlaciona perfectamente consigo misma). Las correlaciones más relevantes para futuras predicciones serán las que involucran `positive_ratings` y `average_playtime`, ya que muestran relaciones interesantes con otras variables del dataset.

---
## 11. Conclusiones

Del análisis exploratorio del dataset de Steam Games se pueden extraer las siguientes conclusiones:

1. **Calidad del dataset:** El dataset está casi completo, con muy pocos o ningún valor nulo, lo que facilita el trabajo de limpieza.

2. **Distribuciones sesgadas:** La mayoría de las variables numéricas (ratings, tiempo de juego, precio) tienen una distribución fuertemente asimétrica: pocos juegos concentran la mayor actividad, mientras que la mayoría tiene valores bajos. Esto es característico del mercado de productos digitales.

3. **Dominio Indie:** El género Indie es el más abundante, reflejando la democratización del desarrollo de videojuegos en los últimos años.

4. **Precio no determina calidad:** No se observa una correlación fuerte entre el precio de un juego y su valoración por los usuarios, lo que sugiere que la calidad percibida depende de otros factores.

5. **Potencial para ML:** Las variables `positive_ratings`, `price`, `genres`, `achievements` y `average_playtime` son buenos candidatos para construir un modelo de clasificación que prediga si un juego tendrá buena recepción por parte del público.